In [104]:
import sim          
import sympy as sp  
import numpy as np
import time
import math
import cv2
import matplotlib.pyplot as plt

def connect(port):
    sim.simxFinish(-1)
    clientID=sim.simxStart('127.0.0.1',port,True,True,2000,5) # Conectarse
    if clientID == 0: print("conectado a", port)
    else: print("no se pudo conectar")
    sim.simxSynchronous(clientID, True)
    sim.simxStartSimulation(clientID, sim.simx_opmode_blocking)
    return clientID

In [105]:
clientID = connect(19999)

retCode,camara=sim.simxGetObjectHandle(clientID,'Vision_sensor',sim.simx_opmode_blocking)

retCode,ruedaDerecha=sim.simxGetObjectHandle(clientID,'RuedaR',sim.simx_opmode_blocking)
retCode,ruedaIzquierda=sim.simxGetObjectHandle(clientID,'RuedaL',sim.simx_opmode_blocking)

retCode, electroiman = sim.simxGetObjectHandle(clientID, 'Electroiman', sim.simx_opmode_blocking)

ret,ultrasonidoDerecha=sim.simxGetObjectHandle(clientID,'SensorR',sim.simx_opmode_blocking)
ret,ultrasonidoIzquierda=sim.simxGetObjectHandle(clientID,'SensorL',sim.simx_opmode_blocking)
ret,ultrasonidoDelante=sim.simxGetObjectHandle(clientID,'SensorD',sim.simx_opmode_blocking)
ret,ultrasonidoAtras=sim.simxGetObjectHandle(clientID,'SensorA',sim.simx_opmode_blocking)

conectado a 19999


In [106]:
def esperarPasos(tiempo):
    pasos = int(tiempo / 0.05)
    for _ in range(pasos):
        sim.simxSynchronousTrigger(clientID)

def controlar_electroiman(estado):
    sim.simxSetIntegerSignal(clientID, "estado_iman", estado, sim.simx_opmode_blocking)

def obtenerDistanciaSensor(ultrasonido):
    errorCode, detectionState, detectedPoint, detectedObjectHandle, detectedSurfaceNormalVector=sim.simxReadProximitySensor(clientID,ultrasonido, sim.simx_opmode_blocking)
    sensor_val=np.linalg.norm(detectedPoint)
    return sensor_val

def aplicarVelocidades(vel_izq, vel_der):
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda, vel_izq, sim.simx_opmode_oneshot)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,   vel_der, sim.simx_opmode_oneshot)

def frenar():
    aplicarVelocidades(0, 0)

In [109]:
def avanzar(velocidad, tiempo):
    v = velocidad
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda, v, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,   v, sim.simx_opmode_blocking)
    esperarPasos(tiempo)
    detener()

def moverCasilla(v_angular):
    v_lineal = v_angular*0.04
    t = 0.25/v_lineal
    t_real = t * 1

    print(f"Tiempo casilla: {t_real}")
    aplicarVelocidades(v_angular, v_angular)
    esperarPasos(t_real)
    frenar()
    

def giro90(v, direccion):
    radianes_rueda = math.radians(90 * direccion) * 0.17 / 0.08
    tiempo = radianes_rueda / v
    
    print(f"tiempo Giro: {tiempo}")
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda, v, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,  -v, sim.simx_opmode_blocking)
    
    esperarPasos(tiempo)
    
    # 5. Frenamos ambos motores
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda,  0, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,    0, sim.simx_opmode_blocking)


def foto():
    retCode, resolution, image=sim.simxGetVisionSensorImage(clientID,camara,0,sim.simx_opmode_oneshot_wait)
    img = np.array(image, dtype=np.float32)
    img = img.astype(np.uint8)  # Convierte a uint8
    img.resize(resolution[1], resolution[0], 3)
    img = cv2.flip(img, 0)
    #img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    plt.imshow(img)
    plt.show()

def girar(v_angular, grados):
    radio_rueda = 0.04  # 4 cm
    L = 0.185            # 18 cm (Distancia entre ruedas)
    
    # Aseguramos que la velocidad angular sea positiva para los cálculos de tiempo
    v_angular_abs = abs(v_angular)
    
    # Calcular la distancia que debe rodar cada rueda (usando L / 2)
    grados_abs = abs(grados)
    radianes_giro = math.radians(grados_abs)
    distancia_rueda = radianes_giro * (L / 2)  # <--- AQUÍ ESTABA EL CAMBIO
    
    # Calcular velocidad lineal de la rueda y el tiempo necesario
    v_lineal = v_angular_abs * radio_rueda
    t = distancia_rueda / v_lineal
    
    # Determinar el sentido de giro según si "grados" es positivo o negativo
    if grados > 0:
        # Giro antihorario (Izquierda atrás, Derecha adelante)
        vel_izq = -v_angular_abs
        vel_der = v_angular_abs
    else:
        # Giro horario (Izquierda adelante, Derecha atrás)
        vel_izq = v_angular_abs
        vel_der = -v_angular_abs

    print(f"tiempo Giro: {t}")
    # Ejecutar el movimiento
    aplicarVelocidades(vel_izq, vel_der)
    esperarPasos(t) 
    frenar()

def calibrate(v_angular_base):
    # Usamos una velocidad muy baja para no pasarnos el mínimo por inercia
    v_lenta = v_angular_base * 0.3 
    
    # Hacemos el giro de prueba inicial (giro horario)
    dist_inicial = obtenerDistanciaSensor(ultrasonidoIzquierda)
    aplicarVelocidades(v_lenta, -v_lenta)
    esperarPasos(0.08)
    frenar()
    esperarPasos(0.02)
    dist_final = obtenerDistanciaSensor(ultrasonidoIzquierda)
    
    error = dist_inicial - dist_final
    print(error)
    # Inicializamos la variable para rastrear la distancia más corta encontrada
    dist_minima = dist_final
    
    if error > 0 +0.00001:
        # Si 'error > 0', significa que dist_inicial > dist_final (la distancia disminuyó).
        # Íbamos por buen camino girando en sentido horario. Continuamos.
        print("Alineando en sentido horario...")
        while True:
            aplicarVelocidades(v_angular_base, -v_angular_base)
            # En modo síncrono, sustituye las dos líneas siguientes por: sim.simxSynchronousTrigger(clientID)
            esperarPasos(0.05) 
            frenar() 
            
            dist_actual = obtenerDistanciaSensor(ultrasonidoIzquierda)
            print(dist_actual)
            # Si la distancia actual es mayor que la mínima que habíamos registrado, 
            # significa que acabamos de pasar el punto perpendicular (el mínimo). ¡Paramos!
            if dist_actual > dist_minima + 0.0001: # Añadimos un pequeño margen de 1mm para evitar ruido
                break
            
            # Si sigue disminuyendo, actualizamos nuestro récord de distancia mínima
            if dist_actual < dist_minima:
                dist_minima = dist_actual
    else:
        # Si 'error <= 0', al girar hacia la derecha la distancia aumentó. 
        # Significa que el coche debía girar hacia el otro lado (antihorario).
        print("Alineando en sentido antihorario...")
        dist_minima = dist_inicial # Reseteamos al valor más bajo conocido
        while True:
            aplicarVelocidades(-v_angular_base, v_angular_base)
            esperarPasos(0.05)
            frenar()
            
            dist_actual = obtenerDistanciaSensor(ultrasonidoIzquierda)
            
            if dist_actual > dist_minima + 0.0001: 
                break
                
            if dist_actual < dist_minima:
                dist_minima = dist_actual

    frenar()
    print("¡Robot alineado de forma perpendicular!")


    
def giroContinuo(v, direccion):
    radianes_rueda = math.radians(90 * direccion) * 0.17 / 0.08
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda,  v, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,   -v, sim.simx_opmode_blocking)

def detener():
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda,  0, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,   0, sim.simx_opmode_blocking)

def movimientoContinuo(velocidad, direccion):
    v = velocidad * direccion
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda, v, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,   v, sim.simx_opmode_blocking)

def pillarLlave():
    controlar_electroiman(1)
    avanzar(1, 1)
    avanzar(-1, 1)


In [110]:
pillarLlave()